# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


In [3]:
print('shape:', df.shape)
print(df.dtypes)
print('nulls per column:')
print(df.isnull().sum())
print('exact duplicate rows:', df.duplicated().sum())

shape: (315, 5)
order_id      int64
item         object
category     object
qty         float64
price        object
dtype: object
nulls per column:
order_id     0
item         0
category     0
qty         13
price        0
dtype: int64
exact duplicate rows: 15


### TODO 1 — drop duplicates

In [4]:
removed= df.duplicated().sum()
clean= df.drop_duplicates().copy()


print('rows:', len(df), '-->', len(clean))

log('duplicates', 'dropped exact duplicate rows', removed)

rows: 315 --> 300
[duplicates] dropped exact duplicate rows (15 row(s))


### TODO 2 — clean `price` -> float

In [5]:
clean['price']= (clean['price'].astype(str).str.replace('$','', regex=False).str.replace(',','',regex=False).str.strip().astype(float))

assert clean['price'].dtype == float

print(clean['price'].dtype)

log('clean price', 'converted price to a float, stripped whitespace and stripped dollar signs', len(clean))


float64
[clean price] converted price to a float, stripped whitespace and stripped dollar signs (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [6]:
clean['qty']= pd.to_numeric(clean['qty'], errors='coerce')
print(clean[['order_id', 'item', 'qty']])
print()

missing= clean['qty'].isna().sum()
negative= (clean['qty']<0).sum()

rev_before = (clean['qty'] * clean['price']).sum()

clean= clean[clean['qty'].notna() & (clean['qty'] > 0)].copy()
clean['qty']= clean['qty'].astype(int)

rev_after = (clean['qty'] * clean['price']).sum()

print(f'rows: {len(df)} -> {len(clean)}')

print(rev_before)
print(rev_after)
print('difference in revenue:', rev_before - rev_after)

log('clean qty', 'cleaned quantity by dropping out rows that have missing or negative quantities of items sold. This gives us a clear idea of the gross sales', len(df)-len(clean))

     order_id           item  qty
0           0    Rain Poncho  3.0
1           1    foam finger  1.0
2           2  cheese burger  1.0
3           3   Cheeseburger  NaN
4           4  cheese burger  1.0
..        ...            ...  ...
295       295    rain poncho  3.0
296       296  cheese burger -1.0
297       297    Foam Finger  1.0
298       298    Rain Poncho  1.0
299       299  cheese burger  1.0

[300 rows x 3 columns]

rows: 315 -> 275
4594.5
4740.0
difference in revenue: -145.5
[clean qty] cleaned quantity by dropping out rows that have missing or negative quantities of items sold. This gives us a clear idea of the gross sales (40 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [7]:
print(df['item'].value_counts())

clean['item'] = (clean['item'].str.lower())

ITEM_MAP= {
    'cheese burger':'cheeseburger'
}

clean['item']= clean['item'].replace(ITEM_MAP)
print('after', sorted(clean['item'].unique()))

log('standardize naming', 'canonicalize item names so that the different spellings are all the same for each of the three items', len(clean))


item
Foam Finger      61
cheese burger    55
Cheeseburger     54
Rain Poncho      52
rain poncho      49
foam finger      44
Name: count, dtype: int64
after ['cheeseburger', 'foam finger', 'rain poncho']
[standardize naming] canonicalize item names so that the different spellings are all the same for each of the three items (275 row(s))


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [8]:
catcopy= clean['category'].copy()
print(df['category'].value_counts())

clean['category']= (clean['category'].str.strip().str.lower().str.replace('-', '', regex=False))

CATEGORY_MAP = {
    'apparel':'merch',
    'raingear':'raingear',
}

clean['category']= clean['category'].replace(CATEGORY_MAP)
print('after:', sorted(clean['category'].unique()))

log('category cleaning', 'classified apparel as a merch item, so both categories now register as merch. normalized case and punctuation', (catcopy != clean['category']).sum())

category
Merch        62
Food         59
food         54
rain-gear    48
Apparel      47
RainGear     45
Name: count, dtype: int64
after: ['food', 'merch', 'raingear']
[category cleaning] classified apparel as a merch item, so both categories now register as merch. normalized case and punctuation (231 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [9]:
assert clean.duplicated().sum() == 0
assert clean['qty'].min() >= 1
assert clean['price'].dtype == float
assert clean['item'].str.islower().all()
assert clean['category'].nunique() == 3
print('clean:', clean.shape)

clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [10]:
clean['revenue']= clean['qty'] * clean['price']

catrevenue= (clean.groupby('category')['revenue'].sum().sort_values(ascending=False))
print(catrevenue.round(2))
print('total revenue:', catrevenue.sum().round(2))

category
food        1656.0
merch       1572.0
raingear    1512.0
Name: revenue, dtype: float64
total revenue: 4740.0


**What I would tell the vendor:** I would tell the vendor to stock more food as it is the best selling category at their store.

### TODO 8 — read back your log

In [11]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicates,dropped exact duplicate rows,15
1,clean price,"converted price to a float, stripped whitespac...",300
2,clean qty,cleaned quantity by dropping out rows that hav...,40
3,standardize naming,canonicalize item names so that the different ...,275
4,category cleaning,"classified apparel as a merch item, so both ca...",231


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.


**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.



 a.
 Step 3 where we dropped rows with negative or non-existent quantity values was the step in which revenue changed the most.
 Before step 3, revenue was 4594.50 dollars and after the step, revenue was 4740.0 dollars. This means revenue rose after the cleaning. This is due to the fact that the negative quantities in the data were being used in the revenue equation and brining negative revenue into the total.

 b.
In Step 3, a reasonable person may have chosen to keep the rows with negative quantites in table. Under the assumption that the negative quantities meant returned items, keeping these rows would allow the viewer to see the net revenue made during the time period captured in the data frame. This would have led total revenue to be lower than it was when I calculated it in step 7.
I chose to drop the rows with negative quantites because I am interested in the gross number of units sold and the gross revenue before we take refunds into account.